In [5]:
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

# Define image size
img_width = 180
img_height = 180

# Define base model (MobileNetV2)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_width, img_height, 3),  # Input shape defined here
    include_top=False,  # Exclude the top layer of the model
    weights='imagenet'
)
base_model.trainable = False  # Freeze base layers initially

# Define input layer
input_layer = keras.Input(shape=(img_width, img_height, 3))

# Define the model using Functional API
x = data_augmentation(input_layer)  # Augment data
x = layers.Rescaling(1./255)(x)  # Rescale image
x = base_model(x)  # Pass through the base model
x = layers.GlobalAveragePooling2D()(x)  # Pooling to reduce dimensions
x = layers.Dropout(0.3)(x)  # Dropout for regularization
x = layers.Dense(256, activation='relu')(x)  # Fully connected layer
x = layers.Dropout(0.5)(x)  # Dropout
output = layers.Dense(len(data_categories), activation='softmax')(x)  # Output layer

# Create the final model
model = keras.Model(inputs=input_layer, outputs=output)

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Summary
model.summary()


C:\Users\HP\AppData\Local\Temp\ipykernel_19484\4014955234.py:10: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)           │ (None, 180, 180, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ sequential (Sequential)              │ (None, 180, 180, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ rescaling_1 (Rescaling)              │ (None, 180, 180, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 6, 6, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_1           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 44)                  │          11,308 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,597,228 (9.91 MB)

 Trainable params: 339,244 (1.29 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [7]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils import class_weight
import numpy as np

# Early stopping callback to stop if validation loss doesn't improve
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=5,  # Stop after 5 epochs with no improvement
    restore_best_weights=True  # Restore the model weights from the best epoch
)

# Learning rate reduction callback to reduce learning rate on plateau
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',  # Monitor validation loss
    factor=0.2,  # Reduce learning rate by 20%
    patience=3,  # Wait for 3 epochs before reducing learning rate
    min_lr=1e-6  # Set minimum learning rate
)

# Assuming you have class weights (optional)
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(data_train.labels),
    y=data_train.labels
)
class_weights = dict(enumerate(class_weights))

# Train the model
history = model.fit(
    data_train,  # Training data
    validation_data=data_val,  # Validation data
    epochs=50,  # Set the number of epochs
    batch_size=32,  # Set the batch size
    callbacks=[early_stopping, reduce_lr],  # Callbacks for early stopping and learning rate reduction
    class_weight=class_weights  # Use class weights if needed
)


AttributeError: '_PrefetchDataset' object has no attribute 'labels'